In [2]:
#code used to clean-up the cpations which are not well formatted
#the detection criteria is : missing rank + 3 consecutive digits separated by commas
#some files are missing the rank column, such as the file 698.csv
import os
import csv

#folder_path = r"C:\Users\hanna\ADA\2025\newyorker_caption_contest\data"
#C:\Users\hanna\ADA\2025\newyorker_caption_contest\data

import os
import csv

input_folder = r"C:\Users\hanna\ADA\2025\newyorker_caption_contest\data"

In [ ]:

MAX_CAPTION_LENGTH = 250  # threshold for corrupted captions

def is_corrupted_row(row, caption_index=1, max_length=MAX_CAPTION_LENGTH):
    """
    Detect corrupted rows:
    - Rows with fewer than 8 columns (malformed)
    - Rows with caption length > max_length
    """
    if len(row) != 8:
        return True  # malformed row
    caption = row[caption_index]
    return len(caption) > max_length

# Iterate over all CSV files
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        input_path = os.path.join(input_folder, filename)

        with open(input_path, "r", encoding="utf-8", newline='') as infile:
            reader = csv.reader(infile)
            next(reader)  # skip header

            for row_number, row in enumerate(reader, start=2):
                if is_corrupted_row(row):
                    print(f"{filename}: corrupted row at line {row_number}")

523.csv: corrupted row at line 492
523.csv: corrupted row at line 1885
523.csv: corrupted row at line 2124
523.csv: corrupted row at line 2499
523.csv: corrupted row at line 2860
523.csv: corrupted row at line 3162
523.csv: corrupted row at line 3238
523.csv: corrupted row at line 3296
523.csv: corrupted row at line 3338
523.csv: corrupted row at line 3853
523.csv: corrupted row at line 4057
531.csv: corrupted row at line 3854
660.csv: corrupted row at line 2
660.csv: corrupted row at line 3
660.csv: corrupted row at line 4
660.csv: corrupted row at line 5
660.csv: corrupted row at line 6
660.csv: corrupted row at line 7
660.csv: corrupted row at line 8
660.csv: corrupted row at line 9
660.csv: corrupted row at line 10
660.csv: corrupted row at line 11
660.csv: corrupted row at line 12
660.csv: corrupted row at line 13
660.csv: corrupted row at line 14
660.csv: corrupted row at line 15
660.csv: corrupted row at line 16
660.csv: corrupted row at line 17
660.csv: corrupted row at line 18

KeyboardInterrupt: 

In [3]:
#checking id headers are right
EXPECTED_HEADERS = ["rank", "caption", "mean", "precision", "votes", "not_funny", "somewhat_funny", "funny"]

def has_correct_headers(headers):
    """
    Returns True if the CSV headers match the expected headers exactly.
    """
    return headers == EXPECTED_HEADERS

# Iterate over all CSV files
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        input_path = os.path.join(input_folder, filename)

        with open(input_path, "r", encoding="utf-8", newline='') as f:
            reader = csv.reader(f)
            try:
                headers = next(reader)
            except StopIteration:
                print(f"{filename}: file is empty")
                continue

            if not has_correct_headers(headers):
                print(f"{filename}: incorrect headers -> {headers}")


660.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
661.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
662.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
663.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
664.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
665.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
666.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
667.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', 'funny']
668.csv: incorrect headers -> ['caption', 'mean', 'precision', 'votes', 'not_funny', 'somewhat_funny', '

In [4]:
MISSING_RANK_HEADERS = ["caption", "mean", "precision", "votes", "not_funny", "somewhat_funny", "funny"]

# Counters
missing_rank_count = 0
other_header_variants = set()

# Iterate over all CSV files
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        input_path = os.path.join(input_folder, filename)

        with open(input_path, "r", encoding="utf-8", newline='') as f:
            reader = csv.reader(f)
            try:
                headers = next(reader)
            except StopIteration:
                continue  # skip empty files

            if headers == MISSING_RANK_HEADERS:
                missing_rank_count += 1
            elif headers != EXPECTED_HEADERS:
                other_header_variants.add(tuple(headers))

print(f"Number of files missing the 'rank' column: {missing_rank_count}")

if other_header_variants:
    print("Other variants of missing headers found:")
    for variant in other_header_variants:
        print(variant)
else:
    print("No other header variants found.")

Number of files missing the 'rank' column: 236
No other header variants found.


In [5]:
#fixing the files with the missing rank header
#testing on 698.csv first
file_to_fix = "698.csv"  # test on this file only

EXPECTED_HEADERS = ["rank", "caption", "mean", "precision", "votes", "not_funny", "somewhat_funny", "funny"]
MISSING_RANK_HEADERS = ["caption", "mean", "precision", "votes", "not_funny", "somewhat_funny", "funny"]

def fix_missing_rank(file_path):
    """Add a rank column based on descending mean values."""
    rows = []

    # Read the CSV
    with open(file_path, "r", encoding="utf-8", newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                row['mean'] = float(row['mean'])  # convert mean to float for sorting
                rows.append(row)
            except ValueError:
                continue  # skip rows with invalid mean

    # Sort rows by mean descending
    rows.sort(key=lambda x: x['mean'], reverse=True)

    # Write back the CSV with rank
    with open(file_path, "w", encoding="utf-8", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(EXPECTED_HEADERS)  # write header

        for i, row in enumerate(rows):
            writer.writerow([
                i,  # rank
                row['caption'],
                row['mean'],
                row['precision'],
                row['votes'],
                row['not_funny'],
                row['somewhat_funny'],
                row['funny']
            ])

# Check and fix only this file
input_path = os.path.join(input_folder, file_to_fix)

with open(input_path, "r", encoding="utf-8", newline='') as f:
    reader = csv.reader(f)
    try:
        headers = next(reader)
    except StopIteration:
        print(f"{file_to_fix} is empty.")
        headers = []

if headers == MISSING_RANK_HEADERS:
    print(f"{file_to_fix} is missing 'rank'. Fixing it now...")
    fix_missing_rank(input_path)
    print(f"{file_to_fix} has been fixed.")
else:
    print(f"{file_to_fix} already has correct headers.")

698.csv is missing 'rank'. Fixing it now...
698.csv has been fixed.
